# Classical Machine Learning (CML) Benchmark
## Multi-Feature Dimensionality Study (4, 6, and 8 Features)
**Experimental Focus:** Lung Cancer Screening Benchmark  
**Dataset:** Survey Lung Cancer Dataset (`V1_dataset.csv`, 2,998 total records, 450 held-out test samples)

---

### 1. Module Objectives
1. Train and evaluate standard classical models (**Logistic Regression**, **Linear SVM**, and **RBF Kernel SVM**) across 4, 6, and 8-feature representations.
2. Establish leakage-free classical baselines to benchmark against Variational Quantum Classifiers (VQCs).
3. Analyze performance trade-offs across Accuracy, Balanced Accuracy, Sensitivity (Recall), Specificity, F1-Score, and ROC-AUC.
4. Follow rigorous scientific interpretation: Metric $\to$ Statistical / Clinical Meaning $\to$ Quantitative Observation $\to$ Practical Takeaway.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
)

sns.set_theme(style="whitegrid", palette="muted")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths setup
notebook_dir = Path.cwd().resolve()
data_dir = notebook_dir.parent / "data" / "processed"

print(f"Data Directory: {data_dir}")
assert (data_dir / "y_test.csv").exists(), "Processed dataset not found!"

### 2. Dataset Ingestion (Leakage-Free Splitting)

We load the standardized feature splits derived from the Survey Lung Cancer Dataset:
- **Training Set**: 2,098 samples (70%)
- **Validation Set**: 450 samples (15%)
- **Holdout Test Set**: 450 samples (15%: 227 cancer positive, 223 negative)
- All continuous attributes were scaled using `MinMaxScaler` fitted exclusively on training data.

In [ ]:
# Load Targets
y_train = pd.read_csv(data_dir / "y_train.csv").squeeze().values.astype(int)
y_val = pd.read_csv(data_dir / "y_validation.csv").squeeze().values.astype(int)
y_test = pd.read_csv(data_dir / "y_test.csv").squeeze().values.astype(int)

# Load 4, 6, and 8-Feature Matrices
X_train_4 = pd.read_csv(data_dir / "X_train_4.csv").values
X_val_4 = pd.read_csv(data_dir / "X_validation_4.csv").values
X_test_4 = pd.read_csv(data_dir / "X_test_4.csv").values

X_train_6 = pd.read_csv(data_dir / "X_train_6.csv").values
X_val_6 = pd.read_csv(data_dir / "X_validation_6.csv").values
X_test_6 = pd.read_csv(data_dir / "X_test_6.csv").values

X_train_8 = pd.read_csv(data_dir / "X_train_8.csv").values
X_val_8 = pd.read_csv(data_dir / "X_validation_8.csv").values
X_test_8 = pd.read_csv(data_dir / "X_test_8.csv").values

print(f"Loaded Datasets:")
print(f"  4 Features: Train {X_train_4.shape}, Test {X_test_4.shape}")
print(f"  6 Features: Train {X_train_6.shape}, Test {X_test_6.shape}")
print(f"  8 Features: Train {X_train_8.shape}, Test {X_test_8.shape}")
print(f"  Test Class Distribution: Positive={sum(y_test==1)}, Negative={sum(y_test==0)}")

### 3. Evaluation Function & Metrics Protocol

We define a standard evaluation function reporting:
- **Accuracy**: Overall fraction of correct classifications.
- **Balanced Accuracy**: Mean of Sensitivity and Specificity, correcting for class distribution.
- **Sensitivity (Recall)**: $\text{TP} / (\text{TP} + \text{FN})$ — percentage of actual cancer cases detected.
- **Specificity**: $\text{TN} / (\text{TN} + \text{FP})$ — percentage of non-cancer individuals correctly identified.
- **Precision**: $\text{TP} / (\text{TP} + \text{FP})$ — reliability of a positive prediction.
- **F1-Score**: Harmonic mean of Precision and Sensitivity.
- **ROC-AUC**: Area Under the Receiver Operating Characteristic curve.

In [ ]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X_te)
    
    acc = accuracy_score(y_te, y_pred)
    bacc = balanced_accuracy_score(y_te, y_pred)
    sens = recall_score(y_te, y_pred, pos_label=1)
    spec = recall_score(y_te, y_pred, pos_label=0)
    prec = precision_score(y_te, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_te, y_pred, pos_label=1, zero_division=0)
    auc = roc_auc_score(y_te, y_prob)
    cm = confusion_matrix(y_te, y_pred)
    
    return {
        'Model Name': name,
        'Accuracy': round(acc, 4),
        'Balanced Accuracy': round(bacc, 4),
        'Sensitivity': round(sens, 4),
        'Specificity': round(spec, 4),
        'Precision': round(prec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc, 4),
        'True Positives': int(cm[1, 1]),
        'False Positives': int(cm[0, 1]),
        'True Negatives': int(cm[0, 0]),
        'False Negatives': int(cm[1, 0])
    }

# Run Benchmark across 4, 6, and 8 Features
models_config = [
    ('Logistic Regression (4 Feats)', LogisticRegression(max_iter=500, random_state=42), X_train_4, X_test_4),
    ('Linear SVM (4 Feats)', SVC(kernel='linear', C=0.01, probability=True, random_state=42), X_train_4, X_test_4),
    ('RBF SVM (4 Feats)', SVC(kernel='rbf', C=0.1, gamma=0.05, probability=True, random_state=42), X_train_4, X_test_4),
    ('Logistic Regression (6 Feats)', LogisticRegression(max_iter=500, random_state=42), X_train_6, X_test_6),
    ('Linear SVM (6 Feats)', SVC(kernel='linear', C=0.01, probability=True, random_state=42), X_train_6, X_test_6),
    ('RBF SVM (6 Feats)', SVC(kernel='rbf', C=0.1, gamma=0.05, probability=True, random_state=42), X_train_6, X_test_6),
    ('Linear SVM (8 Feats)', SVC(kernel='linear', C=0.01, probability=True, random_state=42), X_train_8, X_test_8),
    ('RBF SVM (8 Feats)', SVC(kernel='rbf', C=0.1, gamma=0.05, probability=True, random_state=42), X_train_8, X_test_8)
]

results_list = []
for name, clf, x_tr, x_te in models_config:
    res = evaluate_model(name, clf, x_tr, y_train, x_te, y_test)
    results_list.append(res)

df_results = pd.DataFrame(results_list)
df_results[['Model Name', 'Accuracy', 'Balanced Accuracy', 'Sensitivity', 'Specificity', 'F1-Score', 'ROC-AUC', 'False Positives', 'False Negatives']]

### 4. In-Depth Interpretation of Classical Benchmark Results

#### 1. 4-Feature Configuration (`WHEEZING`, `YELLOW_FINGERS`, `AGE`, `SHORTNESS_OF_BREATH`)
- **Observed Performance**: Accuracy ranged between 50.89% and 52.44%, with balanced accuracy near 51-52% and ROC-AUC around 0.50-0.53.
- **Interpretation**: With only 4 predictors, classical linear and kernel hyperplanes struggle to separate cancer from non-cancer cases because these four symptoms are widespread across non-malignant chronic pulmonary patients.

#### 2. 6-Feature Configuration (`+ COUGHING`, `PEER_PRESSURE`, `CHEST_PAIN`)
- **Sensitivity Improvement**: Logistic Regression achieved **68.28% sensitivity** (155 of 227 positive cases detected) and Classical RBF SVM reached **60.79% sensitivity** with peak accuracy of **54.67%**.
- **Interpretation**: Adding persistent coughing and localized chest pain added genuine discriminatory clinical signal, producing the most effective trade-off among classical algorithms.

#### 3. 8-Feature Configuration (`+ ANXIETY`, `ALLERGY`, `CHRONIC_DISEASE`)
- **Boundary Saturation Phenomenon**: Both Linear and RBF SVMs collapsed into predicting positive for all samples (Sensitivity = 100%, Specificity = 0%, False Positives = 223).
- **Statistical Rationale**: Adding non-specific lifestyle and psychological variables increases the density of overlapping patient symptom profiles. Without re-weighted penalty terms, standard maximum-margin optimization pushes the separation boundary outside the feature cloud, demonstrating that feature expansion without regularization degrades classification quality.